# LangGraph — Human-in-the-Loop (HITL)

এই notebook-এ দেখানো হবে কীভাবে LangGraph-এ **মানুষ AI-এর tool call-এ সিদ্ধান্ত নিতে পারে** — approve, reject, বা modify।

## Human-in-the-Loop কী?

```
Without HITL:
  User: "Dhaka-র আবহাওয়া?"
    ↓
  [chatbot] → get_weather("dhaka") → auto-execute → answer
  (মানুষ কিছু দেখে না)

With HITL:
  User: "Dhaka-র আবহাওয়া?"
    ↓
  [chatbot] → get_weather("dhaka")
    ↓
  ⏸️  PAUSED — Human review!
    ↓
  Human decides:
    approve → tool চলে, original answer
    reject  → tool blocked, different answer
    modify  → city "london" করা হলো → completely different answer!
```

## এই Notebook-এর মূল শিক্ষা

**একই user প্রশ্ন + ভিন্ন human decision = ভিন্ন tool call = ভিন্ন LLM উত্তর**

| Scenario | Human Decision | Tool Call যা চলে | LLM উত্তর |
|---|---|---|---|
| A | ✅ Approve | `get_weather("dhaka")` | ঢাকায় ৩৪°C গরম... |
| B | ❌ Reject | (blocked — চলে না) | আবহাওয়া জানা গেল না... |
| C | ✏️ Modify → London | `get_weather("london")` | লন্ডনে ১৭°C ঠান্ডা... |

এটাই এই notebook-এর core demo।

## কেন HITL দরকার?

| সমস্যা | HITL সমাধান |
|---|---|
| LLM ভুল city assume করে | Human modify করতে পারে |
| Sensitive action auto-run হয় | Human approve না করলে blocked |
| Irreversible action (email, delete) | Execution-এর আগে review |
| Audit trail দরকার | প্রতিটা decision logged থাকে |

---
## মূল Concepts

### ১. `interrupt()` — Graph Pause করে

`interrupt()` হলো LangGraph-এর built-in function যেটা graph execution **পুরোপুরি থামিয়ে** দেয়।

```python
from langgraph.types import interrupt

def human_review_node(state):
    # ── এই line-এ graph PAUSE হয় ──
    decision = interrupt({
        'tool_name': 'get_weather',
        'tool_args': {'city': 'dhaka'},
    })
    # ── resume হলে decision পাওয়া যায় ──
    # decision = {"action": "approve"} বা {"action": "reject"} ইত্যাদি
```

**`interrupt()` → `invoke()` → `Command(resume=...)` flow:**

```
graph.invoke({messages: [HumanMessage(...)]}, config)   ← 1st call
    ↓
[chatbot] runs
    ↓
[human_review] → interrupt() called
    ↓
invoke() RETURNS! (result["__interrupt__"] আছে)         ← control ফিরে আসে
    ↓
Human দেখে tool_name, tool_args → সিদ্ধান্ত নেয়
    ↓
graph.invoke(Command(resume={"action": "approve"}), config)  ← 2nd call
    ↓
[human_review] → interrupt() returns {"action": "approve"}
    ↓
graph continues → [tools] → [chatbot] → END
```

> ⚠️ **interrupt() ব্যবহার করতে `checkpointer` আবশ্যক।** MemorySaver() ছাড়া graph interrupt state save করতে পারে না।

---

### ২. `Command(resume=...)` — Graph Resume করে

```python
from langgraph.types import Command

# Approve — original args দিয়ে tool চলবে
graph.invoke(Command(resume={'action': 'approve'}), config)

# Reject — tool blocked, LLM rejection দেখবে
graph.invoke(Command(resume={'action': 'reject', 'reason': 'Privacy concern'}), config)

# Modify — city বদলে দাও, ভিন্ন tool result আসবে
graph.invoke(Command(resume={'action': 'modify', 'new_args': {'city': 'london'}}), config)
```

---

### ৩. তিনটা Decision এবং Tool Call-এ প্রভাব

| Decision | `human_review_node` কী করে | State পরিবর্তন | Tool যা চলে |
|---|---|---|---|
| **approve** | কিছু করে না | অপরিবর্তিত | Original: `get_weather("dhaka")` |
| **reject** | ToolMessage("rejected") যোগ করে | Rejection message state-এ | চলে না → chatbot rejection দেখে |
| **modify** | AIMessage replace করে নতুন args দিয়ে | পুরনো tool_call → নতুন city | `get_weather("london")` — ভিন্ন! |

---
### Step 1 — Environment Setup

**কী হচ্ছে:** `.env` থেকে `ANTHROPIC_API_KEY` load।

**কেন checkpointer আবশ্যক:** `interrupt()` graph-এর state save করে — checkpointer ছাড়া এটা সম্ভব না। পরে `Command(resume=...)` দিলে সেই saved state থেকে continue করে।

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
print('ANTHROPIC_API_KEY:', bool(ANTHROPIC_API_KEY))

---
### Step 2 — Weather Tool

Notebook 1-এর মতো একই `get_weather` tool — HITL-এ এটাই approve/reject/modify হবে।

**HITL demo scenario:**
- User: *"আমার শহরের আবহাওয়া বলো"*
- LLM assume করে: `get_weather("dhaka")`
- Human দেখে tool call → সিদ্ধান্ত নেয়
  - **approve** → ঢাকার আবহাওয়া
  - **reject** → no weather data
  - **modify** to "london" → লন্ডনের আবহাওয়া (সম্পূর্ণ ভিন্ন উত্তর!)

In [ ]:
from langchain_core.tools import tool

weather_db = {
    'dhaka':      {'temperature': '34°C', 'condition': 'Sunny',         'humidity': '72%', 'wind_speed': '10 km/h', 'feels_like': '38°C'},
    'chittagong': {'temperature': '32°C', 'condition': 'Partly Cloudy', 'humidity': '78%', 'wind_speed': '14 km/h', 'feels_like': '36°C'},
    'london':     {'temperature': '17°C', 'condition': 'Overcast',      'humidity': '85%', 'wind_speed': '20 km/h', 'feels_like': '15°C'},
    'new york':   {'temperature': '22°C', 'condition': 'Clear',         'humidity': '55%', 'wind_speed': '15 km/h', 'feels_like': '21°C'},
}

@tool
def get_weather(city: str) -> str:
    'Return current weather for a given city.'
    key = city.lower().strip()
    if key not in weather_db:
        available = ', '.join(weather_db.keys())
        return f"No weather data for '{city}'. Available: {available}"
    w = weather_db[key]
    return '\n'.join([
        f'Weather in {city.title()}:',
        f'  Temperature : {w["temperature"]} (feels like {w["feels_like"]})',
        f'  Condition   : {w["condition"]}',
        f'  Humidity    : {w["humidity"]}',
        f'  Wind Speed  : {w["wind_speed"]}',
    ])

tools = [get_weather]
print('Tool:', get_weather.name)
print('Args:', get_weather.args)

---
### Step 3 — State এবং LLM

**State:**
- Notebook 1-এর মতোই — `messages: Annotated[list, add_messages]`
- HITL-এ কোনো extra state field লাগে না

**LLM:**
- `bind_tools(tools)` দিয়ে LLM-কে tool-aware করা হচ্ছে
- Claude Haiku ব্যবহার করা হচ্ছে

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langchain_anthropic import ChatAnthropic

class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatAnthropic(
    model='claude-haiku-4-5-20251001',
    anthropic_api_key=ANTHROPIC_API_KEY,
)
llm_with_tools = llm.bind_tools(tools)

print('State :', dict(State.__annotations__))
print('Model :', llm.model)
print('Tools :', [t.name for t in tools])

---
### Step 4 — Nodes তৈরি

**তিনটা node:**

| Node | ধরন | কাজ |
|---|---|---|
| `chatbot` | Custom | LLM call — tool call decide করে |
| `human_review` | Custom + `interrupt()` | Tool call দেখায়, মানুষের সিদ্ধান্ত নেয় |
| `tools` | `ToolNode` (prebuilt) | Tool execute করে |

---

**`human_review_node` — তিনটা decision handle করে:**

```
interrupt() called → graph paused
    ↓
Command(resume={"action": "approve"})
    → return None (nothing changes)
    → route_after_review sees AIMessage with tool_calls → "tools"

Command(resume={"action": "reject", "reason": "..."})
    → return {"messages": [ToolMessage("[REJECTED]...")]}
    → route_after_review sees ToolMessage → "chatbot" (LLM দেখবে, নতুন response দেবে)

Command(resume={"action": "modify", "new_args": {"city": "london"}})
    → return {"messages": [AIMessage(same id, new args)]}
    → add_messages: পুরনো AIMessage replace হয় (same id!)
    → route_after_review sees AIMessage with tool_calls → "tools"
```

**Modify-এর key trick:** নতুন AIMessage তৈরিতে `id=last_msg.id` দিলে `add_messages` reducer পুরনো message **replace** করে (append করে না)। এতে tool call-এর args পরিবর্তন হয়ে যায়।

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.prebuilt import ToolNode
from langgraph.types import interrupt

# ── Node 1: chatbot ──
def chatbot(state: State):
    response = llm_with_tools.invoke(state['messages'])
    return {'messages': [response]}

# ── Node 2: human_review ──
def human_review_node(state: State):
    last_msg = state['messages'][-1]

    # Tool call নেই → review দরকার নেই
    if not (hasattr(last_msg, 'tool_calls') and last_msg.tool_calls):
        return

    tool_call = last_msg.tool_calls[0]

    # ── PAUSE: মানুষের সিদ্ধান্তের জন্য অপেক্ষা ──
    decision = interrupt({
        'tool_name': tool_call['name'],
        'tool_args': tool_call['args'],
        'message'  : f"Tool '{tool_call['name']}' args={tool_call['args']} — approve / reject / modify?",
    })

    action = decision.get('action', 'approve')

    if action == 'approve':
        # কিছু করার নেই — tools node-এ যাবে original args দিয়ে
        pass

    elif action == 'reject':
        reason = decision.get('reason', 'No reason given')
        return {
            'messages': [ToolMessage(
                content=f'[REJECTED] Human blocked tool call. Reason: {reason}',
                tool_call_id=tool_call['id'],
            )]
        }

    elif action == 'modify':
        new_args = decision.get('new_args', tool_call['args'])
        # Updated tool call with new args
        updated_tc = {
            'id'  : tool_call['id'],
            'name': tool_call['name'],
            'args': new_args,
            'type': 'tool_call',
        }
        # Same id → add_messages পুরনো AIMessage REPLACE করে (append না)
        return {
            'messages': [AIMessage(
                content=last_msg.content,
                tool_calls=[updated_tc],
                id=last_msg.id,
            )]
        }

# ── Node 3: tools (prebuilt) ──
tool_node = ToolNode(tools)

print('Nodes defined: chatbot, human_review_node, tool_node')

---
### Step 5 — Graph তৈরি

**Graph flow:**

```
START
  ↓
[chatbot]
  ↓
after_chatbot (conditional edge)
  ├── tool_calls আছে? → [human_review]  ← interrupt() এখানে
  │       ↓
  │   after_review (conditional edge)
  │       ├── approve / modify → [tools] → [chatbot]
  │       └── reject           → [chatbot] ← rejection দেখে নতুন response
  │
  └── tool_calls নেই? → END
```

**দুটো routing function:**

```python
def after_chatbot(state):
    # tool_calls আছে → human_review
    # নেই → END

def after_review(state):
    # reject হলে (ToolMessage) → chatbot
    # approve/modify (AIMessage with tool_calls) → tools
```

In [ ]:
import uuid
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import ToolMessage

# ── Routing: chatbot → human_review বা END ──
def after_chatbot(state: State):
    last_msg = state['messages'][-1]
    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
        return 'human_review'
    return END

# ── Routing: human_review → tools বা chatbot ──
def after_review(state: State):
    last_msg = state['messages'][-1]
    # Rejection: ToolMessage যোগ হয়েছে → chatbot-এ পাঠাও (LLM rejection দেখবে)
    if isinstance(last_msg, ToolMessage):
        return 'chatbot'
    # Approve/Modify: AIMessage with tool_calls → tools চালাও
    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
        return 'tools'
    return END

# ── Graph build ──
builder = StateGraph(State)

builder.add_node('chatbot',      chatbot)
builder.add_node('human_review', human_review_node)
builder.add_node('tools',        tool_node)

builder.add_edge(START, 'chatbot')

builder.add_conditional_edges('chatbot',      after_chatbot)
builder.add_conditional_edges('human_review', after_review)

builder.add_edge('tools', 'chatbot')

# ── interrupt() আবশ্যক: checkpointer দিয়ে compile ──
graph = builder.compile(checkpointer=MemorySaver())

print('Graph compiled!')
print('Nodes:', list(graph.get_graph().nodes.keys()))
print('Edges:')
for e in graph.get_graph().edges:
    print(f'  {e[0]} \u2192 {e[1]}')

---
### Step 6 — Graph Visualization

Graph-এ এখন নতুন `human_review` node দেখা যাবে — chatbot এবং tools-এর মাঝখানে।

In [ ]:
from IPython.display import Image, display

print('=== HITL Graph (Mermaid) ===')
print(graph.get_graph().draw_mermaid())

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f'PNG render skipped: {e}')

---
## Scenario A — ✅ Approve (Tool চলে, Original Answer)

**Flow:**
```
User: "আমার শহরের আবহাওয়া বলো" (LLM assumes Dhaka)
    ↓
[chatbot] → get_weather("dhaka") → PAUSE
    ↓
Human: "হ্যাঁ, ঠিক আছে" → approve
    ↓
[tools] → get_weather("dhaka") → "34°C, Sunny..."
    ↓
[chatbot] → "ঢাকায় ৩৪°C গরম..."
```

**কীভাবে করব:**

```python
# Step 1: প্রথম invoke → interrupt হবে
result = graph.invoke({"messages": [HumanMessage(...)]}, config)
# result["__interrupt__"] আছে — graph paused

# Step 2: tool_call দেখো, সিদ্ধান্ত নাও
# Step 3: approve করো
result = graph.invoke(Command(resume={"action": "approve"}), config)
# এখন graph শেষ হয়েছে
```

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.types import Command

config_a = {'configurable': {'thread_id': str(uuid.uuid4())}}
query = 'আমার শহরের আবহাওয়া বলো। আমি Dhaka-তে থাকি।'

print('=' * 58)
print('Scenario A — APPROVE')
print('=' * 58)
print(f'Human: {query}\n')

# ── Step 1: first invoke → interrupt ──
result_a1 = graph.invoke(
    {'messages': [HumanMessage(content=query)]},
    config=config_a,
)

interrupt_data = result_a1.get('__interrupt__')
if interrupt_data:
    review_info = interrupt_data[0].value
    print('>>> PAUSED — Human Review Required!')
    print(f'    Tool Name : {review_info["tool_name"]}')
    print(f'    Tool Args : {review_info["tool_args"]}')
    print(f'    Message   : {review_info["message"]}')
    print()

    # ── Step 2: Human approves ──
    print('Human decision: APPROVE')
    print('>>> Resuming with approve...\n')

    result_a2 = graph.invoke(
        Command(resume={'action': 'approve'}),
        config=config_a,
    )
    ai_content = result_a2['messages'][-1].content
    if not isinstance(ai_content, str):
        ai_content = str(ai_content)
    print('Final AI Answer:')
    print(ai_content[:400] + ('...' if len(ai_content) > 400 else ''))
else:
    print('No interrupt — check graph setup.')

---
## Scenario B — ❌ Reject (Tool Blocked, Different Answer)

**একই প্রশ্ন, কিন্তু Human reject করলে সম্পূর্ণ ভিন্ন উত্তর।**

**Flow:**
```
User: "আমার শহরের আবহাওয়া বলো" (LLM assumes Dhaka)
    ↓
[chatbot] → get_weather("dhaka") → PAUSE
    ↓
Human: "না, আমি চাই না" → reject (reason: "Privacy")
    ↓
State-এ ToolMessage([REJECTED]) যোগ হয়
    ↓
[chatbot] → ToolMessage দেখে → "আবহাওয়া জানা গেল না..."
    ↓
Same question, DIFFERENT answer!
```

**State change দেখো:**

```
Approve-এর পরে State:
  [HumanMessage]  "আমার শহরের আবহাওয়া বলো"
  [AIMessage]     tool_calls=[get_weather("dhaka")]
  [ToolMessage]   "Weather in Dhaka: 34°C..."
  [AIMessage]     "ঢাকায় ৩৪°C গরম..."

Reject-এর পরে State:
  [HumanMessage]  "আমার শহরের আবহাওয়া বলো"
  [AIMessage]     tool_calls=[get_weather("dhaka")]
  [ToolMessage]   "[REJECTED] Human blocked tool call"
  [AIMessage]     "দুঃখিত, আবহাওয়া জানা গেল না।"  ← DIFFERENT!
```

In [ ]:
config_b = {'configurable': {'thread_id': str(uuid.uuid4())}}

print('=' * 58)
print('Scenario B — REJECT (same question!)')
print('=' * 58)
print(f'Human: {query}\n')   # same query as Scenario A

# ── Step 1: first invoke → interrupt ──
result_b1 = graph.invoke(
    {'messages': [HumanMessage(content=query)]},
    config=config_b,
)

interrupt_data = result_b1.get('__interrupt__')
if interrupt_data:
    review_info = interrupt_data[0].value
    print('>>> PAUSED — Human Review Required!')
    print(f'    Tool Name : {review_info["tool_name"]}')
    print(f'    Tool Args : {review_info["tool_args"]}')
    print()

    # ── Step 2: Human rejects ──
    print('Human decision: REJECT (reason: Privacy concern)')
    print('>>> Resuming with reject...\n')

    result_b2 = graph.invoke(
        Command(resume={'action': 'reject', 'reason': 'Privacy concern'}),
        config=config_b,
    )

    # State inspect করো — rejection দেখো
    state_b = graph.get_state(config_b)
    msgs = state_b.values.get('messages', [])
    print('State after reject:')
    for m in msgs:
        role = type(m).__name__
        content = str(m.content)[:80].replace('\n', ' ')
        print(f'  [{role}] {content}')

    print()
    ai_content = result_b2['messages'][-1].content
    if not isinstance(ai_content, str):
        ai_content = str(ai_content)
    print('Final AI Answer (DIFFERENT from Scenario A!):')
    print(ai_content[:400] + ('...' if len(ai_content) > 400 else ''))
else:
    print('No interrupt.')

---
## Scenario C — ✏️ Modify (Tool Args বদলে যায়, Completely Different Answer)

**এটাই HITL-এর সবচেয়ে powerful feature — Human tool call-এর input পরিবর্তন করে।**

**Flow:**
```
User: "আমার শহরের আবহাওয়া বলো" (LLM assumes Dhaka)
    ↓
[chatbot] → get_weather("dhaka") → PAUSE
    ↓
Human: "না, আমি এখন London-এ আছি" → modify → city: "london"
    ↓
human_review_node:
    পুরনো AIMessage (get_weather "dhaka", id=abc)
    নতুন AIMessage (get_weather "london", id=abc)  ← same id!
    add_messages: পুরনো REPLACE হয়
    ↓
[tools] → get_weather("london") চলে  ← DHAKA না, LONDON!
    ↓
[chatbot] → "লন্ডনে ১৭°C ঠান্ডা..."  ← COMPLETELY DIFFERENT!
```

**Modify-এর State change:**

```
Before modify:
  messages[-1] = AIMessage(tool_calls=[{name:get_weather, args:{city:dhaka}, id:"abc"}])

After modify (same id → replace):
  messages[-1] = AIMessage(tool_calls=[{name:get_weather, args:{city:london}, id:"abc"}])

tools node চলার পরে:
  messages += [ToolMessage("Weather in London: 17°C, Overcast...")]

chatbot-এর final answer:
  "লন্ডনে আজ ১৭°C ঠান্ডা এবং মেঘলা আবহাওয়া..."
```

In [ ]:
config_c = {'configurable': {'thread_id': str(uuid.uuid4())}}

print('=' * 58)
print('Scenario C — MODIFY (city: dhaka → london)')
print('=' * 58)
print(f'Human: {query}\n')   # same query as Scenarios A & B

# ── Step 1: first invoke → interrupt ──
result_c1 = graph.invoke(
    {'messages': [HumanMessage(content=query)]},
    config=config_c,
)

interrupt_data = result_c1.get('__interrupt__')
if interrupt_data:
    review_info = interrupt_data[0].value
    print('>>> PAUSED — Human Review Required!')
    print(f'    Tool Name : {review_info["tool_name"]}')
    print(f'    Tool Args : {review_info["tool_args"]}  ← LLM assumed Dhaka')
    print()

    # ── Step 2: Human modifies city ──
    print('Human decision: MODIFY → city: "london" (আমি এখন London-এ আছি)')
    print('>>> Resuming with modify...\n')

    result_c2 = graph.invoke(
        Command(resume={'action': 'modify', 'new_args': {'city': 'london'}}),
        config=config_c,
    )

    # State inspect — modified tool call দেখো
    state_c = graph.get_state(config_c)
    msgs = state_c.values.get('messages', [])
    print('State after modify + tool execution:')
    for m in msgs:
        role = type(m).__name__
        if hasattr(m, 'tool_calls') and m.tool_calls:
            tc = m.tool_calls[0]
            print(f'  [{role}] tool_calls=[{tc["name"]}({tc["args"]})]  ← MODIFIED!')
        elif hasattr(m, 'name') and m.name:
            print(f'  [ToolMessage] {str(m.content)[:70].replace(chr(10), " ")}')
        else:
            print(f'  [{role}] {str(m.content)[:70].replace(chr(10), " ")}')

    print()
    ai_content = result_c2['messages'][-1].content
    if not isinstance(ai_content, str):
        ai_content = str(ai_content)
    print('Final AI Answer (COMPLETELY DIFFERENT — London, not Dhaka!):')
    print(ai_content[:400] + ('...' if len(ai_content) > 400 else ''))
else:
    print('No interrupt.')

---
## তুলনা: একই প্রশ্ন, তিনটা ভিন্ন Outcome

**Question: "আমার শহরের আবহাওয়া বলো। আমি Dhaka-তে থাকি।"**

| | Scenario A (Approve) | Scenario B (Reject) | Scenario C (Modify) |
|---|---|---|---|
| **Human Action** | ✅ Approve | ❌ Reject | ✏️ city → london |
| **Tool যা চলে** | `get_weather("dhaka")` | (চলে না) | `get_weather("london")` |
| **Tool Result** | 34°C, Sunny | [REJECTED] | 17°C, Overcast |
| **LLM উত্তর** | ঢাকায় ৩৪°C গরম... | আবহাওয়া জানা গেল না... | লন্ডনে ১৭°C ঠান্ডা... |
| **State পরিবর্তন** | ToolMessage (weather) | ToolMessage ([REJECTED]) | AIMessage replaced + ToolMessage |

**মূল শিক্ষা:** Human-এর decision LLM-এর final answer সম্পূর্ণ বদলে দেয় — কারণ tool call-এর input বা execution-ই পরিবর্তন হয়।

---

## interrupt() কখন হয়?

`interrupt()` graph-এর **যেকোনো node-এ** রাখা যায়। এই notebook-এ:

```
[chatbot] → get_weather("dhaka") decide করে
               ↓
         [human_review]  ← interrupt() এখানে
               ↓
         [tools] → tool execute হয়
```

অন্যত্রও রাখা যায়:
- Tools-এর **পরে** (result দেখে approve করতে)
- Chatbot-এর **আগে** (user message screen করতে)
- **Custom logic** node-এ (certain condition হলেই interrupt)

---
## সারসংক্ষেপ

### HITL Graph Structure

```
START → [chatbot]
              ↓ tool_calls আছে?
        [human_review]  ← interrupt() এখানে pause হয়
              ↓
    ┌─────────┴──────────┐
    │ approve/modify      │ reject
    ↓                     ↓
  [tools]           [chatbot] (rejection দেখে)
    ↓                     ↓
  [chatbot]              END
    ↓
   END
```

### Key API

| API | Package | কাজ |
|---|---|---|
| `interrupt(data)` | `langgraph.types` | Graph pause, human-কে `data` দেখাও |
| `Command(resume=val)` | `langgraph.types` | Graph resume, `interrupt()` returns `val` |
| `MemorySaver()` | `langgraph.checkpoint.memory` | interrupt state save করে (আবশ্যক) |

### তিনটা Decision এবং Effect

```python
# Approve
Command(resume={'action': 'approve'})
# → human_review কিছু return করে না
# → tools node original args দিয়ে চলে

# Reject
Command(resume={'action': 'reject', 'reason': '...'})
# → human_review ToolMessage([REJECTED]) return করে
# → chatbot rejection দেখে, ভিন্ন response দেয়

# Modify
Command(resume={'action': 'modify', 'new_args': {'city': 'london'}})
# → human_review AIMessage replace করে (same id!)
# → tools নতুন args দিয়ে চলে → ভিন্ন result → ভিন্ন LLM answer
```

### পরবর্তী Notebook-এ
- **Parallel nodes** — একাধিক node একসাথে চালানো
- **Custom state fields** — messages ছাড়াও extra tracking
- **Multi-agent** — agent-এর মধ্যে agent